# Live Series Simulator

🎯 **Goal**: Apply the simulation framework to a live Taskmaster series, updating win probabilities as new episode data becomes available.

In previous notebooks, I developed a simulation framework to estimate win probabilities across a series.

This notebook applies that approach to a live setting, where episode results are added incrementally and win probabilities are updated over time.

**Approach**:
- Define the expected structure of live episode-level data  
- Aggregate results to build a current performance snapshot for each contestant  
- Estimate scoring behaviour using mean and variability observed so far  
- Simulate remaining episodes using the existing simulation framework  
- Convert simulation outputs into win probabilities  
- Update probabilities as new episode data becomes available  

###  Executive Summary

- Demonstrated how the simulation framework can be applied to a live series  
- Using only episode-level scores, win probabilities can be updated dynamically after each episode  
- Early results can indicate likely winners, but probabilities can shift quickly as more data becomes available  

**Key insight:**  
> The system provides a simple way to track how likely each contestant is to win in real time, updating as the series unfolds.

### Imports

In [ ]:

import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath(".."))

import pandas as pd

from simulation import simulate_many_from_snapshot

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"

PROCESSED_PATH = DATA_DIR / "processed"

### Loading the data

In [2]:
snapshot_df = pd.read_csv(PROCESSED_PATH / "snapshot_df.csv")

snapshot_df.head()

,episode,episode_label,contestant,episode_score,series_id,episode_in_series,cumulative_score,mean_score_so_far,std_score_so_far,recent_avg_score,momentum,episodes_played,final_score,total_episodes,final_rank,won_series,remaining_episodes,current_rank
0,1,Melon buffet.,Frank Skinner,19,1,1,19,19.0,0.0,19.0,0.0,1,93,6,2.0,0,5,1.0
1,1,Melon buffet.,Josh Widdicombe,13,1,1,13,13.0,0.0,13.0,0.0,1,94,6,1.0,1,5,4.0
2,1,Melon buffet.,Roisin Conaty,7,1,1,7,7.0,0.0,7.0,0.0,1,68,6,5.0,0,5,5.0
3,1,Melon buffet.,Romesh Ranganathan,19,1,1,19,19.0,0.0,19.0,0.0,1,93,6,2.0,0,5,1.0
4,1,Melon buffet.,Tim Key,17,1,1,17,17.0,0.0,17.0,0.0,1,88,6,4.0,0,5,3.0


## The Simulator

### Define live data model

In [ ]:
live_results = pd.DataFrame({
    "contestant": ["Amy Gledhill", "Armando Iannucci", "Joanna Page", "Joel Dommett", "Kumail Nanjiani"],
    "episode": [1, 1, 1, 1, 1],
    "episode_score": [5, 4, 3, 2, 1],   # EXAMPLE SCORES
})

live_results

,contestant,episode,episode_score
0,Amy Gledhill,1,5
1,Armando Iannucci,1,4
2,Joanna Page,1,3
3,Joel Dommett,1,2
4,Kumail Nanjiani,1,1


### Building a dynamic snapshot

In [ ]:
live_snapshot = (
    live_results
    .sort_values(["contestant", "episode"])
    .groupby("contestant")
    .agg({
        "episode_score": ["sum", "mean", "std", "count"]
    })
)

live_snapshot.columns = [
    "cumulative_score",
    "mean_score_so_far",
    "std_score_so_far",
    "episodes_played"
]

live_snapshot = live_snapshot.reset_index()

# Add remaining episodes (assuming 10 total)
total_episodes = 10

live_snapshot["remaining_episodes"] = (
    total_episodes - live_snapshot["episodes_played"]
)

global_std = snapshot_df["episode_score"].std()

live_snapshot["std_score_so_far"] = live_snapshot["std_score_so_far"].fillna(global_std)

live_snapshot.loc[
    live_snapshot["std_score_so_far"] == 0,
    "std_score_so_far"
] = global_std

live_snapshot

,contestant,cumulative_score,mean_score_so_far,std_score_so_far,episodes_played,remaining_episodes
0,Amy Gledhill,5,5.0,4.242711,1,9
1,Armando Iannucci,4,4.0,4.242711,1,9
2,Joanna Page,3,3.0,4.242711,1,9
3,Joel Dommett,2,2.0,4.242711,1,9
4,Kumail Nanjiani,1,1.0,4.242711,1,9


### Run live simulation

In [ ]:
live_simulations = simulate_many_from_snapshot(
    live_snapshot,
    n_simulations=1000)

live_simulations.head()

,contestant,final_score,simulation_id,rank
0,Amy Gledhill,58.961537,0,1
1,Joanna Page,41.322842,0,2
2,Kumail Nanjiani,33.682652,0,3
3,Joel Dommett,18.142340,0,4
4,Armando Iannucci,17.844789,0,5


In [12]:
# Calculate win probabilities
live_win_probabilities = (
    live_simulations[live_simulations["rank"] == 1]
    .groupby("contestant")
    .size()
    .reset_index(name="wins")
)

live_win_probabilities["win_probability"] = (
    live_win_probabilities["wins"] / 1000
)

# Merge with all contestants to include those with zero wins
all_contestants = pd.DataFrame({
    "contestant": live_snapshot["contestant"]
})

live_win_probabilities = all_contestants.merge(
    live_win_probabilities,
    on="contestant",
    how="left"
)

live_win_probabilities["wins"] = live_win_probabilities["wins"].fillna(0)
live_win_probabilities["win_probability"] = live_win_probabilities["win_probability"].fillna(0)

# Sort by win probability
live_win_probabilities = live_win_probabilities.sort_values(
    "win_probability", ascending=False
).reset_index(drop=True)

live_win_probabilities

,contestant,wins,win_probability
0,Amy Gledhill,637,0.637
1,Armando Iannucci,271,0.271
2,Joanna Page,71,0.071
3,Joel Dommett,20,0.020
4,Kumail Nanjiani,1,0.001


#### ⚠️ Example live output

Using example episode results, the simulation produces updated win probabilities based on performance observed so far.

In a live setting, these scores would be replaced with actual episode results as each new episode airs.

## Week by week updates

In [14]:
# Add dummy data for episode 2
# Combines episode 1 and episode 2 results for each contestant
live_results_ep2 = pd.DataFrame({
    "contestant": ["Amy Gledhill", "Armando Iannucci", "Joanna Page", "Joel Dommett", "Kumail Nanjiani", "Amy Gledhill", "Armando Iannucci", "Joanna Page", "Joel Dommett", "Kumail Nanjiani"],
    "episode": [1, 1, 1, 1, 1, 2, 2, 2, 2, 2],
    "episode_score": [5, 4, 3, 2, 1, 4, 5, 2, 3, 1],   # EXAMPLE SCORES
})

In [15]:
# Rebuild live snapshot with episode 2 data
live_snapshot_ep2 = (
    live_results_ep2
    .sort_values(["contestant", "episode"])
    .groupby("contestant")
    .agg({
        "episode_score": ["sum", "mean", "std", "count"]
    })
)

live_snapshot_ep2.columns = [
    "cumulative_score",
    "mean_score_so_far",
    "std_score_so_far",
    "episodes_played"
]

live_snapshot_ep2 = live_snapshot_ep2.reset_index()

# Add remaining episodes (assuming 10 total)
total_episodes = 10

live_snapshot_ep2["remaining_episodes"] = (
    total_episodes - live_snapshot_ep2["episodes_played"]
)

global_std = snapshot_df["episode_score"].std()

live_snapshot_ep2["std_score_so_far"] = live_snapshot_ep2["std_score_so_far"].fillna(global_std)

live_snapshot_ep2.loc[
    live_snapshot_ep2["std_score_so_far"] == 0,
    "std_score_so_far"
] = global_std

live_snapshot_ep2

,contestant,cumulative_score,mean_score_so_far,std_score_so_far,episodes_played,remaining_episodes
0,Amy Gledhill,9,4.5,0.707107,2,8
1,Armando Iannucci,9,4.5,0.707107,2,8
2,Joanna Page,5,2.5,0.707107,2,8
3,Joel Dommett,5,2.5,0.707107,2,8
4,Kumail Nanjiani,2,1.0,4.242711,2,8


In [16]:
# Re-run simulations with episode 2 data
live_simulations_ep2 = simulate_many_from_snapshot(
    live_snapshot_ep2,
    n_simulations=1000)

live_simulations_ep2.head()

,contestant,final_score,simulation_id,rank
0,Amy Gledhill,46.991181,0,1
1,Armando Iannucci,39.998559,0,2
2,Joanna Page,26.073267,0,3
3,Joel Dommett,23.498058,0,4
4,Kumail Nanjiani,22.150050,0,5


In [17]:
# Re-calculate win probabilities
live_win_probabilities_ep2 = (
    live_simulations_ep2[live_simulations_ep2["rank"] == 1]
    .groupby("contestant")
    .size()
    .reset_index(name="wins")
)

live_win_probabilities_ep2["win_probability"] = (
    live_win_probabilities_ep2["wins"] / 1000
)

# Merge with all contestants to include those with zero wins
all_contestants = pd.DataFrame({
    "contestant": live_snapshot_ep2["contestant"]
})

live_win_probabilities_ep2 = all_contestants.merge(
    live_win_probabilities_ep2,
    on="contestant",
    how="left"
)

live_win_probabilities_ep2["wins"] = live_win_probabilities_ep2["wins"].fillna(0)
live_win_probabilities_ep2["win_probability"] = live_win_probabilities_ep2["win_probability"].fillna(0)

# Sort by win probability
live_win_probabilities_ep2 = live_win_probabilities_ep2.sort_values(
    "win_probability", ascending=False
).reset_index(drop=True)

live_win_probabilities_ep2

,contestant,wins,win_probability
0,Armando Iannucci,507.0,0.507
1,Amy Gledhill,488.0,0.488
2,Kumail Nanjiani,5.0,0.005
3,Joanna Page,0.0,0.000
4,Joel Dommett,0.0,0.000


### 📊 Comparing updates across episodes

⚠️ **EXAMPLE SCORES ONLY**

| contestant | ep1_prob | ep2_prob |
| ---------- | -------- | -------- |
| Amy        | 0.637    | 0.488    |
| Armando    | 0.271    | 0.507    |
| …          | …        | …        |

#### 🔍 How probabilities update with new data

After Episode 1, Amy Gledhill appears as a strong early favourite, with a win probability of over 60%. However, after Episode 2, the picture changes significantly, with Armando Iannucci becoming the most likely winner and probabilities between the top two contestants converging.

This illustrates how early signals can be informative but unstable, and how win probabilities update dynamically as more performance data becomes available.

## 🚀 How this would work in practice

Once a new Taskmaster series starts, I can update the input table after each episode with the latest scores, rebuild the live snapshot, and rerun the simulation.

This would produce an updated set of win probabilities each week, turning the project into a lightweight live forecasting system.

## 💡 Conclusion

This notebook demonstrates how the simulation framework can be applied in a live setting.

By updating contestant performance after each episode and re-running the simulation, it is possible to track evolving win probabilities throughout a series. This transforms the project from retrospective analysis into a lightweight forecasting tool.

The example highlights how early signals can shift as more data becomes available, reinforcing the importance of modelling uncertainty alongside performance.

This approach provides a practical and interpretable way to understand competition dynamics in real time.

👉 With real episode data, this framework could be used to produce weekly updates as a series airs.